# Semana 31 · ¿Qué color tiene una estrella?

Este cuaderno acompaña a la sesión de proyecto de la semana 31: la curva de Planck, las leyes de Wien y de Stefan-Boltzmann, el color de un cuerpo negro y cómo se estima la temperatura de una estrella con dos filtros.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. La curva de Planck

$$B_\lambda(T) = \frac{2hc^2}{\lambda^5}\;\frac{1}{e^{\,hc/(\lambda k_B T)} - 1}.$$

$B_\lambda$ es la potencia emitida por unidad de superficie, de longitud de onda y de ángulo sólido. Las unidades no importan para lo que hacemos: comparamos formas y posiciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

h, c, kB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
sigma = 5.670374e-8

def planck(lam_m, T):
    x = h * c / (lam_m * kB * T)
    with np.errstate(over="ignore"):                  # para x muy grande, e^x desborda y la curva vale 0
        return 2 * h * c**2 / lam_m**5 / np.expm1(x)  # expm1(x) = e^x - 1, más preciso para x pequeño

lam = np.linspace(100e-9, 2500e-9, 2000)
plt.figure(figsize=(10, 5))
plt.axvspan(380, 750, color="gold", alpha=0.15, label="visible")
for T in [3000, 4000, 5772, 8000, 12000]:
    plt.plot(lam * 1e9, planck(lam, T), label=f"{T} K")
plt.xlabel("longitud de onda (nm)"); plt.ylabel("B_λ (W/m²/m/sr)")
plt.title("Curvas de Planck"); plt.legend(); plt.grid(alpha=0.3); plt.show()

Las curvas calientes están por encima de las frías en **todas** las longitudes de onda, y su máximo se desplaza hacia el azul. En escala lineal, la de 12 000 K aplasta a las demás. En escala log-log se ven todas, desde una persona hasta una estrella caliente:

In [ ]:
lam = np.logspace(-8, -3, 2000)      # de 10 nm a 1 mm
plt.figure(figsize=(10, 5))
for T in [310, 1000, 3000, 5772, 12000, 30000]:
    plt.loglog(lam * 1e9, planck(lam, T), label=f"{T} K")
plt.axvspan(380, 750, color="gold", alpha=0.2)
plt.ylim(1e-2, 1e18); plt.xlabel("longitud de onda (nm)"); plt.ylabel("B_λ")
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()

Fíjate en el lado derecho de las curvas: todas acaban siendo rectas paralelas, de pendiente $-4$. Es la zona en la que manda la potencia (aproximación de Rayleigh-Jeans, $B_\lambda \propto T/\lambda^4$). En el lado izquierdo, la exponencial las hunde.

## 2. La ley de Wien

Buscamos el máximo de cada curva con `np.argmax` y lo comparamos con $\lambda_\text{máx} = 2{,}898\times10^{-3}/T$.

In [ ]:
lam = np.logspace(-8, -2, 200000)
print(f"{'T (K)':>8}{'máximo (num.)':>16}{'Wien':>14}")
for T in [310, 2800, 3600, 5772, 9900, 12000]:
    lam_max = lam[np.argmax(planck(lam, T))]
    print(f"{T:>8}{lam_max * 1e9:>13.1f} nm{2.898e-3 / T * 1e9:>11.1f} nm")

Y la constante de Wien a partir de la ecuación $x = 5(1 - e^{-x})$, por iteración de punto fijo (apartado opcional de la semana):

In [ ]:
x = 5.0
for paso in range(6):
    x = 5 * (1 - np.exp(-x))
    print(f"paso {paso + 1}: x = {x:.6f}")
print(f"constante de Wien = hc/(x·kB) = {h * c / (x * kB):.4e} m·K")

## 3. La ley de Stefan-Boltzmann

La potencia que sale de cada metro cuadrado de superficie es $\pi$ veces el área bajo la curva (el factor $\pi$ sale de sumar sobre todas las direcciones de salida). La integramos numéricamente con la regla del trapecio (semana 11) y comparamos con $\sigma T^4$.

In [ ]:
integrar = getattr(np, "trapezoid", None) or np.trapz       # el nombre cambió en NumPy 2
lam = np.logspace(-9, -1, 200000)
Ts = np.array([300, 1000, 3000, 5772, 10000, 30000])
P = np.array([np.pi * integrar(planck(lam, T), lam) for T in Ts])
for T, p in zip(Ts, P):
    print(f"T = {T:>6} K: integral = {p:.4e} W/m²   σT⁴ = {sigma * T**4:.4e} W/m²")

pend = np.polyfit(np.log10(Ts), np.log10(P), 1)[0]
plt.figure(figsize=(6, 4.5))
plt.loglog(Ts, P, "o-")
plt.xlabel("T (K)"); plt.ylabel("potencia por m² (W)"); plt.title(f"pendiente log-log = {pend:.3f}")
plt.grid(alpha=0.3, which="both"); plt.show()

x = np.linspace(1e-6, 60, 200000)
print(f"∫ x³/(eˣ-1) dx = {integrar(x**3 / np.expm1(x), x):.6f}   π⁴/15 = {np.pi**4 / 15:.6f}")

## 4. El color de un cuerpo negro

Reutilizamos lo de la semana 26: las funciones de igualación de color de la CIE (aproximación de Wyman, Sloan y Shirley, 2013) y el paso a colores de pantalla sRGB. Normalizamos el brillo para quedarnos solo con el tono. Los colores son aproximados y dependen de tu pantalla; el blanco de referencia de sRGB corresponde a unos 6500 K, así que el Sol sale ligeramente cálido.

In [ ]:
def g(x, mu, s1, s2):
    s = np.where(x < mu, s1, s2)
    return np.exp(-0.5 * ((x - mu) / s) ** 2)

def cie_xyz(lam_nm):
    x = 1.056*g(lam_nm, 599.8, 37.9, 31.0) + 0.362*g(lam_nm, 442.0, 16.0, 26.7) - 0.065*g(lam_nm, 501.1, 20.4, 26.2)
    y = 0.821*g(lam_nm, 568.8, 46.9, 40.5) + 0.286*g(lam_nm, 530.9, 16.3, 31.1)
    z = 1.217*g(lam_nm, 437.0, 11.8, 36.0) + 0.681*g(lam_nm, 459.0, 26.0, 13.8)
    return x, y, z

def color_cuerpo_negro(T):
    lam_nm = np.linspace(380, 750, 400)
    esp = planck(lam_nm * 1e-9, T)
    X, Y, Z = [integrar(esp * f, lam_nm) for f in cie_xyz(lam_nm)]
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    rgb = np.clip(M @ np.array([X, Y, Z]), 0, None)
    rgb = rgb / rgb.max()
    return np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb ** (1 / 2.4) - 0.055)

Ts = np.linspace(1500, 20000, 300)
franja = np.array([color_cuerpo_negro(T) for T in Ts])[None, :, :]
fig, ax = plt.subplots(figsize=(11, 1.6))
ax.imshow(franja, aspect="auto", extent=(Ts[0], Ts[-1], 0, 1))
ax.set_yticks([]); ax.set_xlabel("temperatura (K)"); ax.set_title("Color de un cuerpo negro")
plt.tight_layout(); plt.show()

In [ ]:
estrellas = [("Antares", 3500), ("Betelgeuse", 3600), ("Aldebarán", 3900), ("Arturo", 4300),
             ("Sol", 5772), ("Proción", 6500), ("Vega", 9600), ("Sirio", 9900), ("Rigel", 12000)]
fig, ax = plt.subplots(figsize=(12, 2.6), facecolor="black")
for i, (nombre, T) in enumerate(estrellas):
    ax.add_patch(plt.Circle((i, 0), 0.38, color=color_cuerpo_negro(T)))
    ax.text(i, -0.6, f"{nombre}\n{T} K", ha="center", va="top", color="white", fontsize=9)
ax.set_xlim(-0.6, len(estrellas) - 0.4); ax.set_ylim(-1.2, 0.5); ax.set_aspect("equal"); ax.axis("off")
plt.show()

## 5. La temperatura con dos filtros

Los astrónomos no necesitan medir toda la curva de Planck para estimar la temperatura de una estrella. Basta con medir su brillo a través de dos filtros de colores, por ejemplo uno azul (centrado en unos 440 nm) y otro verde-amarillo (unos 550 nm), y calcular el cociente. Cuanto más caliente, más azul en proporción. Lo hacemos aquí de forma simplificada, como si cada filtro solo dejara pasar su longitud de onda central.

El cociente depende solo de la temperatura (no de la distancia ni del tamaño de la estrella, que multiplican las dos medidas por lo mismo). Calculamos esa relación y la invertimos con una búsqueda por **bisección**: partimos un intervalo por la mitad una y otra vez, quedándonos con la mitad donde está la solución.

In [ ]:
def cociente_azul_verde(T):
    return planck(440e-9, T) / planck(550e-9, T)

def temperatura_desde_cociente(R, T_min=2000, T_max=50000):
    for _ in range(60):
        T_med = (T_min + T_max) / 2
        if cociente_azul_verde(T_med) < R:     # el cociente crece con la temperatura
            T_min = T_med
        else:
            T_max = T_med
    return T_med

Ts = np.linspace(2500, 30000, 400)
plt.figure(figsize=(8, 4.5))
plt.plot(Ts, cociente_azul_verde(Ts))
plt.axhline(1, color="gray", ls="--")
plt.xlabel("temperatura (K)"); plt.ylabel("brillo azul / brillo verde"); plt.grid(alpha=0.3)
plt.title("Cociente entre dos filtros en función de la temperatura"); plt.show()

for R in [0.5, 1.0, 1.5]:
    print(f"cociente {R}: T ≈ {temperatura_desde_cociente(R):,.0f} K".replace(",", " "))

**Ejercicio.** Una estrella misteriosa da 2,3 veces más luz en el filtro verde que en el azul. ¿Qué temperatura tiene? ¿De qué color la verías?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
T = temperatura_desde_cociente(1 / 2.3)
print(f"T ≈ {T:.0f} K: una estrella anaranjada, parecida a Betelgeuse o Antares")

## 6. Experimenta

1. Dibuja las curvas de Planck de una persona (310 K) y del fondo cósmico (2,725 K) en la misma gráfica log-log. ¿Cuántas veces más energía por m² emite una persona?
2. Con la ley de Stefan-Boltzmann, calcula el radio de una estrella de 10 000 K que tenga la misma luminosidad que el Sol.
3. En el apartado 5, usa filtros más separados (400 y 700 nm). ¿La relación se vuelve más o menos sensible a la temperatura? ¿Y para estrellas muy calientes?

In [ ]:
# Tu experimento aquí